# tfm3lab — run on Colab (GPU)

Bootstrap only: clone the repo, sync the CUDA environment, run the GPU-hungry scripts.
All the real logic lives in `scripts/*.py` and `src/tfm3lab/` — this notebook has no
experiment code of its own, so a bug fixed once in the repo is fixed everywhere.

Prereqs before running this: accept the license at
https://huggingface.co/google/timesfm-3.0-pytorch, and have a Hugging Face token with
read access ready to paste into `hf auth login` below.

Runtime: make sure Colab is set to a GPU runtime (Runtime -> Change runtime type -> T4 or better).

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip -q install uv
REPO_URL = "https://github.com/IrfEazy/timesfm3-talk"  # e.g. git@github.com:you/timesfm3-talk.git
!git clone -q $REPO_URL timesfm3-talk
%cd timesfm3-talk

/content/timesfm3-talk


In [ ]:
!uv sync --extra cuda

Using CPython 3.12.14
Creating virtual environment at: .venv
Resolved 118 packages in 0.97ms
Prepared 92 packages in 1m 18s                                           
Installed 92 packages in 898ms                              
 + anyio==4.14.2
 + backports-zstd==1.7.0
 + beautifulsoup4==4.15.0
 + brotli==1.2.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + contourpy==1.3.3
 + coverage==7.16.0
 + curl-cffi==0.16.2
 + cycler==0.12.1
 + filelock==3.32.5
 + fonttools==4.64.0
 + formulaic==1.2.2
 + fsspec==2026.7.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.29.0
 + idna==3.19
 + ijson==3.5.1
 + inflate64==1.0.4
 + iniconfig==2.3.0
 + interface-meta==2.0.1
 + jinja2==3.1.6
 + kiwisolver==1.5.1
 + lxml==6.1.2
 + markupsafe==3.0.3
 + matplotlib==3.11.1
 + mpmath==1.3.0
 + multitasking==0.0.13
 + multivolumefile==0.2.3
 + narwhals==2.25.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.4.5.8
 + nv

In [ ]:
# One-time: accept the gated checkpoint's license at the URL above, then authenticate.
!uv run hf auth login

⠸  (21/21)                                                                      Uninstalled 3 packages in 253ms
Installed 21 packages in 455ms                              
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token

Aborted!


## Optional: share data/ and results/ with your local machine via Drive

Skip this cell to just use Colab's local (ephemeral) disk instead.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["TFM3LAB_DATA_ROOT"] = "/content/drive/MyDrive/timesfm3-talk-data"

Mounted at /content/drive


## 1. Fetch data

The full MTG backfill (TCGCSV, ~2.5 years) is the slow part on first run — everything
downloaded is cached, so re-running this cell later only fetches what's missing.

In [ ]:
!uv run scripts/00_probe_tcgcsv.py
!uv run scripts/01_fetch_data.py

1. Resolving 7 DEFAULT_CARDS against the live catalog...
                  label  group_id  product_id group_abbreviation                  product_name
          Ragavan [MH2]      2809      239857                MH2      Ragavan, Nimble Pilferer
      Urza's Saga [MH2]      2809      238619                MH2                   Urza's Saga
        Sheoldred [DMU]      3102      282800                DMU     Sheoldred, the Apocalypse
     The One Ring [LTR]     23019      487805                LTR                  The One Ring
Orcish Bowmasters [LTR]     23019      498367                LTR             Orcish Bowmasters
      Chatterfang [MH2]      2809      239444                MH2 Chatterfang, Squirrel General
 Mishra's Factory [MH2]      2809      239686                MH2              Mishra's Factory

2. Checking the earliest known archive date (2024-02-08) is reachable...
   OK.
3. Checking the day BEFORE that (2024-02-07) is correctly absent...
   OK — confirms TCGCSV_ARCHIVE_ST

## 2-5. Run the experiments

In [ ]:
!uv run scripts/02_exp_mtg.py

Loaded 7 series, 937 days each: ['Chatterfang [MH2]', "Mishra's Factory [MH2]", 'Orcish Bowmasters [LTR]', 'Ragavan [MH2]', 'Sheoldred [DMU]', 'The One Ring [LTR]', "Urza's Saga [MH2]"]
846 origins per series (context_len=64, max_horizon=28)




--- transform: raw ---
  univariate...
  multivariate...

--- transform: log1p ---
  univariate...
/content/timesfm3-talk/.venv/lib/python3.12/site-packages/statsmodels/tsa/holtwinters/model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
  multivariate...
/content/timesfm3-talk/.venv/lib/python3.12/site-packages/statsmodels/tsa/holtwinters/model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)

Wrote 663264 prediction rows, 784 accuracy rows, 784 calibration rows to /content/timesfm3-talk/results


In [ ]:
!uv run scripts/03_exp_shock.py

Loaded 4 market series, 2175 trading days (2018-01-02 .. 2026-09-01)

Validating known events against the data-driven shock detector:
  Crollo Covid (2020-03-16, pre_cutoff): OK
  Invasione Ucraina (2022-02-24, pre_cutoff): NOT DETECTED (using the known date anyway)
  Stretta inflazionistica (2022-06-13, pre_cutoff): NOT DETECTED (using the known date anyway)
  Unwind carry trade yen (2024-08-05, post_cutoff): NOT DETECTED (using the known date anyway)
  Shock dazi (2025-04-03, post_cutoff): OK

Event: Crollo Covid (2020-03-16, pre_cutoff)

Event: Invasione Ucraina (2022-02-24, pre_cutoff)

Event: Stretta inflazionistica (2022-06-13, pre_cutoff)

Event: Unwind carry trade yen (2024-08-05, post_cutoff)

Event: Shock dazi (2025-04-03, post_cutoff)

--- Headline: pre vs post-cutoff, mean adaptation lag (x1.5) ---
arm
post_cutoff    9.5
pre_cutoff     2.0

Wrote 1520 rows, 40 accuracy rows, 30 lag rows to /content/timesfm3-talk/results


In [ ]:
# Pure re-analysis of 02/03's cached predictions — no GPU needed, but harmless to run here too.
!uv run scripts/04_exp_calibration.py

MTG (calm, by construction): 165816 rows
Market, split by proximity to a known event (threshold=3d):
regime
calm     620
shock    140

Calibration curve (empirical vs nominal coverage):
regime             calm     shock
nominal_level                    
0.1            0.165739  0.171429
0.2            0.254590  0.271429
0.3            0.334074  0.378571
0.4            0.408752  0.407143
0.5            0.484396  0.500000
0.6            0.558185  0.592857
0.7            0.630885  0.671429
0.8            0.705983  0.728571
0.9            0.800938  0.835714

Pinball / P10-P90 coverage / mean PIT by regime:
regime      n  pinball_avg  coverage_p10_p90  pit_mean
  calm 166436     0.832338          0.635199  0.513960
 shock    140    14.536731          0.664286  0.495601

Wrote calibration curve + summary to /content/timesfm3-talk/results


In [ ]:
!uv run scripts/05_exp_covariates.py

=== Legitimate covariate 1: market day-of-week ===
  day_of_week: MAE without=70.7521, with=68.0324, relative=0.962

=== Legitimate covariate 2: MTG days-until-next-set-release ===
  3 release dates found for the tracked sets
  days_to_release: MAE without=0.2578, with=0.2694, relative=1.045

=== NEGATIVE CONTROL: leaking the actual future into the covariate ===
  MAE clean=0.2578, MAE leaked=0.2602

Wrote covariate experiment results to /content/timesfm3-talk/results


## Bring results home

`results/*.parquet` is everything the local machine needs — figures, slides, and the demo
notebook all read only from there. Download the `results/` folder (or, if you mounted
Drive above, just `git pull`/sync it back into your local checkout) and commit it.

In [ ]:
import shutil
shutil.make_archive("/content/results", "zip", "results")
from google.colab import files
files.download("/content/results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cp -r results /content/drive/MyDrive/timesfm3-talk-data/results_backup